In [5]:
import helpers as hp
import importlib, eda as de
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
importlib.reload(de) 
import os

In [6]:
x_train, x_test, y_train, train_ids, test_ids = hp.load_csv_data("dataset")

## Abstract

- Remove meta-features & remove survey codes
- Identify feature's categories
- Combine dependant features
- Standardization for continuous features
- One-hot encode for categorical features

### 1- Remove meta-features & remove survey codes

In [7]:
feature_names_train = np.genfromtxt("dataset/x_train.csv", delimiter=",", dtype=str, max_rows=1)
feature_names_without_meta_train = feature_names_train[27:]

feature_names_test = np.genfromtxt("dataset/x_test.csv", delimiter=",", dtype=str, max_rows=1)
feature_names_without_meta_test = feature_names_test[27:]

In [8]:
x_train_1 = de.clean_survey_codes(x_train[:, 27:]  , feature_names=feature_names_without_meta_train, verbose=False)
x_test_1 = de.clean_survey_codes(x_test[:, 27:]  , feature_names=feature_names_without_meta_test, verbose=False)

### 2- Identify feature's categories

In [9]:
categorical_train, continuous_train, binary_train = de.detect_feature_types_refined(x_train_1)
categorical_test, continuous_test, binary_test = de.detect_feature_types_refined(x_test_1)


### 3- Combine dependent categorical features

In [10]:
x_disc_train = x_train_1[:, categorical_train]
names_disc_train = [f for f, m in zip(feature_names_without_meta_train, categorical_train) if m]

x_disc_test = x_test_1[:, categorical_test]
names_disc_test = [f for f, m in zip(feature_names_without_meta_test, categorical_test) if m]


# 2. 100k lines
idx_train = np.random.choice(x_disc_train.shape[0], 100000, replace=False)
idx_test = np.random.choice(x_disc_test.shape[0], 100000, replace=False)

x_sample_train = x_disc_train[idx_train]
x_sample_test = x_disc_test[idx_test]


groups_train, deps_train = de.detect_hierarchical_dependencies(x_sample_train, names_disc_train, verbose=False)
groups_test, deps_test = de.detect_hierarchical_dependencies(x_sample_test, names_disc_test, verbose=False)

In [11]:
cleaned_groups_train, to_remove_train, encoding_plan_train = de.process_dependency_groups(
    x_train_1,
    feature_names_without_meta_train,
    groups_train,                   
    verbose=True
)

cleaned_groups_test, to_remove_test, encoding_plan_test = de.process_dependency_groups(
    x_test_1,
    feature_names_without_meta_test,
    groups_test,                   
    verbose=False
)



=== Dependency Group Processing Summary ===
Removed 0 constant features:

Optimized to 7 dependency groups:
  MEDCOST → USENOW3
  BPHIGH4 → DECIDE → DIFFALON → DRNK3GE5 → FRUIT1 → FRUITJU1 → INCOME2 → MAXDRNKS → SMOKDAY2 → SMOKE100
  ALCDAY5 → BPMEDS
  HEIGHT3 → WEIGHT2
  BLIND → USEEQUIP
  DIFFDRES → DIFFWALK
  EXERANY2 → FVORANG

Encoding plan created for 7 root features.



In [12]:
print(encoding_plan_train)

{np.str_('MEDCOST'): [np.str_('USENOW3')], np.str_('BPHIGH4'): [np.str_('DECIDE'), np.str_('DIFFALON'), np.str_('DRNK3GE5'), np.str_('FRUIT1'), np.str_('FRUITJU1'), np.str_('INCOME2'), np.str_('MAXDRNKS'), np.str_('SMOKDAY2'), np.str_('SMOKE100')], np.str_('ALCDAY5'): [np.str_('BPMEDS')], np.str_('HEIGHT3'): [np.str_('WEIGHT2')], np.str_('BLIND'): [np.str_('USEEQUIP')], np.str_('DIFFDRES'): [np.str_('DIFFWALK')], np.str_('EXERANY2'): [np.str_('FVORANG')]}


In [13]:

dependents_train = [d for deps in encoding_plan_train.values() for d in deps]
dependents_test = [d for deps in encoding_plan_test.values() for d in deps]


# retirer les dépendantes mais garder les racines (et toutes les autres catégorielles)
categorical_train = [
    i for i in categorical_train
    if feature_names_without_meta_train[i] not in dependents_train
]
categorical_test = [
    i for i in categorical_test
    if feature_names_without_meta_test[i] not in dependents_train
]

print(len(categorical_train))

65


### 4- Standardization

In [14]:
x_train_cont_std, x_test_cont_std = de.standardize(x_train_1[:, continuous_train], x_test_1[:, continuous_test])

print(x_train_cont_std.shape, x_test_cont_std.shape)


    

(328135, 74) (109379, 74)


### One-hot encoding for categorical features

In [15]:
x_train_cat_encoded, cat_feature_names_train, skipped_features_train = de.one_hot_encode_numpy(
    x_train_1[:, categorical_train],
    feature_names=[feature_names_without_meta_train[i] for i in categorical_train],
    max_categories=30  # can adjust regarding the dataset
)


x_test_cat_encoded, cat_feature_names_test, skipped_features_test = de.one_hot_encode_numpy(
    x_test_1[:, categorical_test],
    feature_names=[feature_names_without_meta_test[i] for i in categorical_test],
    max_categories=30  # can adjust regarding the dataset
)
print(x_train_1[:, categorical_train].shape)
print("Encoded shape:", x_train_cat_encoded.shape)
print("Encoded shape:", x_test_cat_encoded.shape)

#print("Encoded columns:", cat_feature_names)
#print("Skipped features:", skipped_features)

(328135, 65)
Encoded shape: (328135, 347)
Encoded shape: (109379, 347)


### Concatenate everything to get our cleaned data

In [16]:
# ============================================================
# 🔹 FINAL PREPROCESSING PIPELINE
# ============================================================

print("Train continuous shape:", x_train_cont_std.shape)
print("Test continuous shape :", x_test_cont_std.shape)


# --- 2️⃣ Binary features: Keep as-is
x_train_bin = x_train_1[:, binary_train]
x_test_bin  = x_test_1[:, binary_test]

print("✅ Binary features selected.")
print("Binary shape (train):", x_train_bin.shape)


# --- 4️⃣ Encode test categorical features with the same categories as the train
unique_per_feature = [
    np.unique(x_train_1[:, j][~np.isnan(x_train_1[:, j])])
    for j in categorical_train
]

encoded_blocks_test = []
for j, uniques in enumerate(unique_per_feature):
    col = x_test_1[:, categorical_train[j]]
    encoded = np.zeros((col.shape[0], len(uniques)))
    for i, val in enumerate(uniques):
        mask = (col == val)
        encoded[mask, i] = 1.0
    encoded_blocks_test.append(encoded)

x_test_cat_encoded = np.concatenate(encoded_blocks_test, axis=1)

print("✅ Test categorical one-hot encoded (using train categories).")
print("Test categorical shape:", x_test_cat_encoded.shape)


# --- 5️⃣ Concatenate all blocks
x_train_final = np.concatenate([x_train_cont_std, x_train_cat_encoded, x_train_bin], axis=1)
x_test_final  = np.concatenate([x_test_cont_std, x_test_cat_encoded, x_test_bin], axis=1)

print("============================================================")
print("✅ Final preprocessing complete.")
print("Final training shape:", x_train_final.shape)
print("Final testing shape :", x_test_final.shape)
print("============================================================")


Train continuous shape: (328135, 74)
Test continuous shape : (109379, 74)
✅ Binary features selected.
Binary shape (train): (328135, 88)
✅ Test categorical one-hot encoded (using train categories).
Test categorical shape: (109379, 347)
✅ Final preprocessing complete.
Final training shape: (328135, 509)
Final testing shape : (109379, 509)


In [19]:
from sklearn.utils import resample

# Suppose que y_train est ton vecteur cible et x_train ton tableau de features
X = x_train_final
y = y_train

# Combiner
data = np.column_stack((X, y))
majority = data[y == -1]
minority = data[y == 1]


# Sur-échantillonnage de la minorité
minority_upsampled = resample(minority, replace=True,
                              n_samples=len(majority), random_state=42)

balanced = np.vstack((majority, minority_upsampled))
np.random.shuffle(balanced)

x_balanced = balanced[:, :-1]
y_balanced = balanced[:, -1]


In [20]:
os.makedirs("dataset_clean", exist_ok=True)

np.savetxt("dataset_clean/x_train.csv", x_balanced, delimiter=",")
np.savetxt("dataset_clean/x_test.csv", x_test_final, delimiter=",")
np.savetxt("dataset_clean/y_train.csv", y_balanced, delimiter=",")

In [18]:
np.unique(y, return_counts=True)


(array([-1,  1]), array([299160,  28975]))